# 05 — Training Runs (DQN and DoubleDQN)
This notebook trains both supported agents using `src.workflows.train_workflow`, evaluates them with `src.workflows.evaluate_workflow`, and validates generated artifacts.

Hyperparameters come from `configs/agents/*.yaml` and `configs/base/training.yaml`, with optional notebook overrides for faster reproducible runs.

In [1]:
from __future__ import annotations

from pathlib import Path
import random
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from src.utils.config_loader import resolve_config, load_yaml
from src.utils.seed import set_global_seed
from src.utils.artifact_manager import ArtifactManager
from src.workflows.data_workflow import run_data_workflow
from src.workflows.train_workflow import run_train_workflow
from src.workflows.evaluate_workflow import run_evaluate_workflow
from src.workflows.benchmark_workflow import run_benchmark_workflow

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise FileNotFoundError('Repository root not found.')

ROOT = find_repo_root(Path.cwd())
BASE_CFG = resolve_config(root=str(ROOT))
SEED = int(BASE_CFG.get('training', {}).get('random_seed', 42))
set_global_seed(SEED, deterministic_torch=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PAIR = BASE_CFG['data']['pairs'][0]
OUTPUTS_ROOT = str(ROOT / 'outputs')
am = ArtifactManager(OUTPUTS_ROOT)
print(f'PAIR={PAIR}, SEED={SEED}, OUTPUTS_ROOT={OUTPUTS_ROOT}')

c:\Users\Nabeel\miniconda3\envs\torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-13 05:50:26 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=True)


PAIR=EURUSD, SEED=42, OUTPUTS_ROOT=c:\Users\Nabeel\Desktop\frl-trading-framework\outputs


## Training configuration and deterministic overrides
The base YAML values are preserved, but we apply optional notebook overrides to keep iterations practical while still exercising checkpoints, learning updates, and evaluation paths.

In [5]:
FAST_OVERRIDES = {
    'training': {
        'total_timesteps': 6000,
        'warmup_steps': 500,
        'checkpoint_interval': 2000,
        'evaluation_interval': 2000,
        'resume': {'enabled': False},
    },
    'agent': {
        'training': {'learn_start_steps': 500, 'learn_frequency': 2},
        'exploration': {'epsilon_decay_steps': 5000},
    },
}

def build_agent_config(agent_name: str) -> dict:
    cfg = resolve_config(
        root=str(ROOT),
        agent_config=f'configs/agents/{agent_name}.yaml',
        cli_overrides=deepcopy(FAST_OVERRIDES),
    )
    return cfg

display(pd.DataFrame([
    {'override_key': 'training.total_timesteps', 'value': FAST_OVERRIDES['training']['total_timesteps']},
    {'override_key': 'training.warmup_steps', 'value': FAST_OVERRIDES['training']['warmup_steps']},
    {'override_key': 'training.checkpoint_interval', 'value': FAST_OVERRIDES['training']['checkpoint_interval']},
    {'override_key': 'agent.training.learn_start_steps', 'value': FAST_OVERRIDES['agent']['training']['learn_start_steps']},
]))

,override_key,value
0,training.total_timesteps,6000
1,training.warmup_steps,500
2,training.checkpoint_interval,2000
3,agent.training.learn_start_steps,500


## Train and evaluate DQN

In [6]:
dqn_cfg = build_agent_config('dqn')
dqn_data = run_data_workflow(dqn_cfg, pairs=[PAIR], root=str(ROOT))
dqn_train_df = dqn_data[PAIR]['train']
dqn_test_df = dqn_data[PAIR]['test']

dqn_train_summary = run_train_workflow(
    config=dqn_cfg,
    pair=PAIR,
    train_df=dqn_train_df,
    outputs_root=OUTPUTS_ROOT,
    run_tag='notebook_core_05',
)
dqn_run_dir = am.agent_result_dir('dqn', PAIR) / 'notebook_core_05'

dqn_eval_metrics = run_evaluate_workflow(
    config=dqn_cfg,
    pair=PAIR,
    eval_df=dqn_test_df,
    run_dir=dqn_run_dir,
    checkpoint_path=str(dqn_run_dir / 'checkpoints' / 'checkpoint_latest.pt'),
    split_name='test',
)

for path in [
    dqn_run_dir / 'checkpoints' / 'checkpoint_latest.pt',
    dqn_run_dir / 'checkpoints' / 'training_state.json',
    dqn_run_dir / 'models' / 'model_final.pt',
    dqn_run_dir / 'metrics' / 'train' / 'reward_curve.csv',
    dqn_run_dir / 'metrics' / 'train' / 'loss_curve.csv',
]:
    assert path.exists(), f'Missing expected DQN artifact: {path}'

display(pd.DataFrame([{'agent': 'dqn', **dqn_eval_metrics}]))

2026-03-13 04:42:48 | src.utils.seed | INFO | Global seed set to 42 (deterministic_torch=False)


,agent,cumulative_return,annualized_return,annualized_volatility,max_drawdown,sharpe_ratio,sortino_ratio,win_rate,total_trades,turnover,avg_pyramid_steps,avg_martingale_steps,liquidation_count,liquidation_rate
0,dqn,-0.036043,-0.106272,0.018634,0.041949,-6.020208,-8.089811,0.410256,1326,113.211896,0.0,0.0,0,0.0


## Train and evaluate DoubleDQN

In [ ]:
ddqn_cfg = build_agent_config('doubledqn')
ddqn_data = run_data_workflow(ddqn_cfg, pairs=[PAIR], root=str(ROOT))
ddqn_train_df = ddqn_data[PAIR]['train']
ddqn_test_df = ddqn_data[PAIR]['test']

ddqn_train_summary = run_train_workflow(
    config=ddqn_cfg,
    pair=PAIR,
    train_df=ddqn_train_df,
    outputs_root=OUTPUTS_ROOT,
    run_tag='notebook_core_05',
)
ddqn_run_dir = am.agent_result_dir('doubledqn', PAIR) / 'notebook_core_05'

ddqn_eval_metrics = run_evaluate_workflow(
    config=ddqn_cfg,
    pair=PAIR,
    eval_df=ddqn_test_df,
    run_dir=ddqn_run_dir,
    checkpoint_path=str(ddqn_run_dir / 'checkpoints' / 'checkpoint_latest.pt'),
    split_name='test',
)

for path in [
    ddqn_run_dir / 'checkpoints' / 'checkpoint_latest.pt',
    ddqn_run_dir / 'checkpoints' / 'training_state.json',
    ddqn_run_dir / 'models' / 'model_final.pt',
    ddqn_run_dir / 'metrics' / 'train' / 'reward_curve.csv',
    ddqn_run_dir / 'metrics' / 'train' / 'loss_curve.csv',
]:
    assert path.exists(), f'Missing expected DoubleDQN artifact: {path}'

display(pd.DataFrame([{'agent': 'doubledqn', **ddqn_eval_metrics}]))

## Baseline validation against benchmark policy
We run a random-policy benchmark on the same pair and compare core metrics to ensure trained agents are at least sanity-checked before full experiments.

In [ ]:
benchmark_cfg = load_yaml(ROOT / 'configs' / 'benchmarks' / 'random_policy.yaml')
baseline_metrics = run_benchmark_workflow(
    config=BASE_CFG,
    benchmark_name='random_policy',
    benchmark_config=benchmark_cfg,
    pairs=[PAIR],
    data={PAIR: dqn_test_df},
    outputs_root=OUTPUTS_ROOT,
    split_name='test',
)[PAIR]

comparison_df = pd.DataFrame([
    {'agent': 'random_policy', **baseline_metrics},
    {'agent': 'dqn', **dqn_eval_metrics},
    {'agent': 'doubledqn', **ddqn_eval_metrics},
])

# Soft sanity check (warn instead of hard-fail)
for agent_name, metrics in [('dqn', dqn_eval_metrics), ('doubledqn', ddqn_eval_metrics)]:
    if metrics.get('cumulative_return', -1e9) < baseline_metrics.get('cumulative_return', -1e9):
        print(f'⚠️ {agent_name} cumulative return below random baseline in this short run.')

display(comparison_df[['agent', 'cumulative_return', 'sharpe_ratio', 'max_drawdown', 'turnover']])
print('✅ Training notebook pipeline complete (train, eval, checkpoints, baseline comparison).')